#  MCP — Model Context Protocol
### Giving Agents Access to External Tools via a Standard Protocol

---

## What We've Built So Far

| Notebook | What you learned |
|----------|------------------|
| 1 — LangChain | Prompts, chains, parsers, memory |
| 2 — LangGraph + Tools | `@tool`, graphs, ReAct agent pattern |
| **3 — MCP** | **Standardized protocol for external tool servers** |

---

## The Problem MCP Solves

In previous notebooks, every tool was a **Python function living inside your project**:

```
your_agent.py
  └── @tool calculator()
  └── @tool lookup_cve()
  └── @tool get_current_time()
```

This works fine for small projects. But what happens when:
- You want to **reuse the same tools across multiple agents**?
- A teammate builds a tool in a **different language** (Node.js, Go)?
- You want to use tools from a **third-party service** without writing the integration yourself?

**MCP (Model Context Protocol)** — published by Anthropic in late 2024 — defines a standard way for tools to live in **separate servers** that any AI client can connect to:

```
  ┌──────────────┐        MCP         ┌──────────────────────┐
  │  AI Client   │ ◄────────────────► │     MCP Server       │
  │  (your agent)│   standard protocol│  (tools live here)   │
  └──────────────┘                    └──────────────────────┘
```

The client and server can be:
- On the **same machine** (stdio transport — subprocess)
- On **different machines** (SSE/HTTP transport)
- Written in **different languages**

---

## What We Will Cover Now

1. Write a simple MCP server with tools
2. Start it in a terminal (SSE/HTTP transport — works reliably on Windows)
3. Connect to it from the notebook
4. Use its tools inside a LangGraph agent

---

## Transport: Why We Use SSE (HTTP) Here

MCP supports two transports:

| Transport | How it works | Best for |
|-----------|-------------|----------|
| **stdio** | Client launches server as subprocess, talks via stdin/stdout | Simple scripts, CLI tools |
| **SSE (HTTP)** | Server runs independently, client connects over HTTP | Jupyter, shared servers, production |

We use **SSE** because stdio has known issues on Windows inside Jupyter notebooks (the subprocess can't attach to Jupyter's virtual stderr stream). SSE is also closer to how you'd deploy MCP in a real environment.

In [ ]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets

In [ ]:
import os 
os.environ["LANGCHAIN_TRACING_V2"] = "false"

---
## Section 1: Writing the MCP Server

An MCP server is a **standalone Python script**. It:
- Creates a `FastMCP` app
- Registers tools with `@mcp.tool()`
- Starts listening for connections

Compare this to the previous notebook where you used `@tool` to define tools inside your agent script. The concept is identical — the difference is that here the tools live in their **own process**, accessible to any client over HTTP.

The cell below writes the server to a file. Read through it — it's deliberately simple.

In [ ]:
server_code = '''
#!/usr/bin/env python3
"""
simple_mcp_server.py

A minimal MCP server with three tools.
Run it from a terminal with:
    python simple_mcp_server.py

It will start listening on http://localhost:8000
and your notebook connects to http://localhost:8000/sse
"""

import hashlib
import json
import re
from mcp.server.fastmcp import FastMCP

# --- Create the server ---
mcp = FastMCP(name="Simple Security Server")


# --- Tool 1: Password strength checker ---
@mcp.tool()
def check_password_strength(password: str) -> str:
    """
    Checks the strength of a password.
    Returns a score from 0-100 and suggestions for improvement.
    Use this when the user asks to evaluate or rate a password.
    """
    score = 0
    tips = []

    if len(password) >= 16:
        score += 30
    elif len(password) >= 12:
        score += 20
    elif len(password) >= 8:
        score += 10
    else:
        tips.append("Use at least 8 characters")

    if re.search(r"[A-Z]", password): score += 15
    else: tips.append("Add uppercase letters")

    if re.search(r"[a-z]", password): score += 15
    else: tips.append("Add lowercase letters")

    if re.search(r"[0-9]", password): score += 15
    else: tips.append("Add numbers")

    if re.search(r"[!@#$%^&*()\\-_=+]", password): score += 25
    else: tips.append("Add special characters like !@#$%")

    rating = "Very Weak" if score < 30 else "Weak" if score < 50 else "Moderate" if score < 70 else "Strong" if score < 85 else "Very Strong"

    return json.dumps({
        "score": score,
        "rating": rating,
        "tips": tips or ["No improvements needed!"]
    })


# --- Tool 2: Hash generator ---
@mcp.tool()
def generate_hash(text: str, algorithm: str = "sha256") -> str:
    """
    Generates a cryptographic hash of the input text.
    Supported algorithms: md5, sha1, sha256, sha512.
    Use when the user wants to hash a string or compare hash algorithms.
    """
    try:
        h = hashlib.new(algorithm.lower())
        h.update(text.encode())
        return json.dumps({"algorithm": algorithm.upper(), "hash": h.hexdigest()})
    except ValueError:
        return f"Unsupported algorithm: {algorithm}. Choose from: md5, sha1, sha256, sha512"


# --- Tool 3: OWASP Top 10 reference ---
@mcp.tool()
def get_owasp_top10() -> str:
    """
    Returns the OWASP Top 10 web application security risks (2021 edition).
    Use when the user asks about common web vulnerabilities or security best practices.
    """
    top10 = [
        {"rank": 1, "name": "Broken Access Control"},
        {"rank": 2, "name": "Cryptographic Failures"},
        {"rank": 3, "name": "Injection"},
        {"rank": 4, "name": "Insecure Design"},
        {"rank": 5, "name": "Security Misconfiguration"},
        {"rank": 6, "name": "Vulnerable and Outdated Components"},
        {"rank": 7, "name": "Identification and Authentication Failures"},
        {"rank": 8, "name": "Software and Data Integrity Failures"},
        {"rank": 9, "name": "Security Logging and Monitoring Failures"},
        {"rank": 10, "name": "Server-Side Request Forgery (SSRF)"},
    ]
    return json.dumps(top10)


# --- Start the server ---
if __name__ == "__main__":
    # SSE transport: runs an HTTP server on port 8000
    # Clients connect to http://localhost:8000/sse
    mcp.run(transport="sse")
'''

with open("simple_mcp_server.py", "w") as f:
    f.write(server_code)

print("Server written to simple_mcp_server.py")
print()
print("Before running the next cells, open a NEW terminal and run:")
print("   python simple_mcp_server.py")
print()
print("   You should see something like:")
print("   INFO:     Started server process")
print("   INFO:     Uvicorn running on http://127.0.0.1:8000")
print()
print("   Leave that terminal running and come back here.")

---
## Start the Server First

Before continuing, make sure the server is running in a separate terminal:

```bash
python simple_mcp_server.py
```

You should see uvicorn output confirming it's listening on port 8000. Once that's running, come back and execute the cells below.

---
## Section 2: Connecting and Discovering Tools

With the server running, we create a `MultiServerMCPClient` and point it at the server's SSE endpoint. When we call `get_tools()`, the client:

1. Connects to the server over HTTP
2. Asks it: *what tools do you have?*
3. Returns them as standard **LangChain Tool objects**

Those tool objects are identical to what you got from `@tool` in Notebook 2 — same interface, same behaviour. The agent won't know or care that they came from a remote server.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Point the client at our running server
# SSE transport: just a URL — no subprocess issues
client = MultiServerMCPClient(
    {
        "security": {
            "url": "http://localhost:8000/sse",
            "transport": "sse",
        }
    }
)

print("=> Client configured — pointing at http://localhost:8000/sse")

In [ ]:
# Fetch the tool list from the server
# These come back as standard LangChain Tool objects
mcp_tools = await client.get_tools()

print(f"=> Discovered {len(mcp_tools)} tools:\n")
for tool in mcp_tools:
    print(f"  🔧 {tool.name}")
    print(f"     {tool.description[:90]}..." if len(tool.description) > 90 else f"     {tool.description}")
    print()

In [ ]:
# Call tools directly — just like in Notebook 2
# The fact that they're on a remote server is completely transparent
import json

pwd_tool = next(t for t in mcp_tools if "password" in t.name)
hash_tool = next(t for t in mcp_tools if "hash" in t.name)

# Test password checker
result = await pwd_tool.ainvoke({"password": "hello123"})
print("Weak password:")
print(json.dumps(json.loads(result), indent=2))
print()

result = await pwd_tool.ainvoke({"password": "$uper-S3cure!2024"})
print("Strong password:")
print(json.dumps(json.loads(result), indent=2))

---
## Section 3: Plugging MCP Tools into a LangGraph Agent

This is the payoff. We'll build the same **ReAct agent** from Notebook 2 — but feed it tools that came from the MCP server instead of local `@tool` functions.

The graph code is **identical** to what we saw in the previous notebook. Only the source of `tools` changes.

```
Before:  tools = [calculator, lookup_cve, get_current_time]   ← local functions
Now:  tools = await client.get_tools()                      ← from MCP server
```

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI

# --- State (same as Notebook 2) ---
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# --- Tools from MCP (already fetched above) ---
tools = mcp_tools

# --- LLM ---

llm = ChatOpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                 model="gpt-4o-mini",   # Which model to use
                 temperature=0,        # 0 = deterministic, 1 = creative
                 api_key='any value',
                 default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

llm_with_tools = llm.bind_tools(tools)

# --- Nodes (same as Notebook 2) ---
def agent_node(state: AgentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def router(state: AgentState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END

# --- Graph (same as Notebook 2) ---
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", router, {"tools": "tools", END: END})
builder.add_edge("tools", "agent")

agent = builder.compile()
print("Agent compiled with MCP tools")

In [ ]:
async def ask(question: str):
    print(f"\n{'='*60}")
    print(f"USER: {question}")
    print(f"{'='*60}")
    final_state = await agent.ainvoke(
        {"messages": [HumanMessage(content=question)]}
    )
    print(f"\nAGENT: {final_state['messages'][-1].content}")

In [ ]:
# The agent decides when to call MCP tools — same as in the previous notebook
await ask("Is 'Winter2024!' a strong password? What specific improvements would you suggest?")

In [ ]:
await ask("What are the top 3 OWASP risks I should focus on for a login page?")

In [ ]:
await ask(
    "Generate the SHA-256 hash of the string 'my-api-secret', "
    "and explain why SHA-256 is better than MD5 for this purpose."
)

---
## Section 4: Side-by-Side Comparison

Here's the key insight from all three notebooks laid out clearly:

```
┌─────────────────────────────────────────────────────────────┐
│  Local tools                                                │
│                                                             │
│  @tool                           agent.py                   │
│  def calculator(): ...    ──►    tools = [calculator, ...]  │
│  def lookup_cve(): ...           llm.bind_tools(tools)      │
│                                                             │
│  Everything in one process.                                 │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│  MCP tools                                                  │
│                                                             │
│  simple_mcp_server.py            notebook                   │
│  @mcp.tool()              ──►    tools = await              │
│  def check_password(): ...           client.get_tools()     │
│  def generate_hash(): ...        llm.bind_tools(tools)      │
│                                                             │
│  Tools in their own process, connected over HTTP.           │
│  The agent code is identical.                               │
└─────────────────────────────────────────────────────────────┘
```

### What MCP Buys You

| Capability | Without MCP | With MCP |
|-----------|-------------|----------|
| Share tools across projects | Copy-paste code | Point at the same server URL |
| Use tools from other teams | Rewrite in Python | Connect to their MCP server |
| Update a tool | Redeploy your agent | Update the server independently |
| Multiple agents using same tools | Duplicate code everywhere | All connect to one server |
| Language of tools | Must be Python | Any language that speaks MCP |

---
## Section 5: The MCP Ecosystem

The real value of MCP being an **open standard** is that there's a growing library of pre-built servers you can connect to without writing any tool code at all.

```python
# Example: connect to the official GitHub MCP server
# (requires Node.js and a GitHub token)
client = MultiServerMCPClient({
    "github": {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-github"],
        "transport": "stdio",
        "env": {"GITHUB_PERSONAL_ACCESS_TOKEN": "your-token"}
    }
})
# Your agent can now list repos, open issues, read files, create PRs...
```

Some popular community servers:

| Server | What it gives your agent |
|--------|-------------------------|
| `server-filesystem` | Read/write local files |
| `server-github` | GitHub repos, issues, PRs |
| `server-postgres` | Query a PostgreSQL database |
| `server-brave-search` | Live web search |
| `server-slack` | Read channels, send messages |

Browse the full [**Model Context Protocol servers**](https://github.com/modelcontextprotocol/servers)

---
## Summary

| Concept | What it is | Code |
|---------|-----------|------|
| `FastMCP` | Quick way to build an MCP server | `mcp = FastMCP(name="...")` |
| `@mcp.tool()` | Register a tool on the server | Decorator on any function |
| `mcp.run(transport="sse")` | Start the server over HTTP | Bottom of server script |
| `MultiServerMCPClient` | Connect to one or more MCP servers | Pass a config dict with URL |
| `client.get_tools()` | Fetch tools as LangChain Tool objects | `await client.get_tools()` |
| `ToolNode(tools)` | Plug tools into LangGraph | Same as Notebook 2 |


```